In [3]:
import pandas as pd
df = pd.read_csv('DT_coding_sample_training.csv')

def createBranches(att):
    return df[att].unique().tolist()

In [4]:
print(createBranches('packetSize'))

['Small', 'Large', 'Medium']


In [26]:
def divideByLabel(path):
    subset = df[path]
    counts = {'Y': int((subset['malicious'] == 'Yes').sum()),'N': int((subset['malicious'] == 'No').sum())}
    return counts

In [46]:
condition = (df['source'] == 'Asia') & (df['whiteListedIP'] == 'Yes')
print(divideByLabel(condition))

{'Y': 3, 'N': 0}


In [28]:
import math
def calcEntropy(aDict):
    total = sum(aDict.values())
    if total == 0:
        return 0.0
    entropy = 0.0
    for count in aDict.values():
        if count > 0:
            part = count/total
            entropy -= part * math.log2(part)
    return round(entropy, 4)

In [29]:
calcEntropy({'Y': 3, 'N': 0})

0.0

In [30]:
def findSplitAtt(branch, attList):
    current_counts = divideByLabel(branch)
    current_entropy = calcEntropy(current_counts)
    total_samples = sum(current_counts.values())

    if total_samples == 0:
        return attList[0] if attList else None
    best_gain = -1.0
    best_att = None

    for att in attList:
        branches = createBranches(att)
        weighted_entropy = 0.0

        for val in branches:
            sub_path = branch & (df[att] == val)

            sub_counts = divideByLabel(sub_path)
            sub_total = sum(sub_counts.values())

            if sub_total > 0:
                weight = sub_total/total_samples
                weighted_entropy += weight * calcEntropy(sub_counts)
            gain = current_entropy - weighted_entropy

            if gain > best_gain:
                best_gain = gain
                best_att = att
        return best_att

In [31]:
root_path = pd.Series([True] * len(df))
features = ['source', 'whiteListedIP', 'packetSize', 'appType']
print(findSplitAtt(root_path,features))

source


In [32]:
def checkStopping(branch, attList):
    count = divideByLabel(branch)
    total_samples = sum(count.values())
    if total_samples == 0:
        return 'stop_empty'
    if count['Y'] == total_samples or count['N'] == total_samples:
        return 'stop_pure'
    if len(attList) == 0:
        return 'no_attributes_left'
    return 'continue'

In [33]:
pure_path = (df['source'] == 'Asia') & (df['whiteListedIP'] == 'Yes')
print(checkStopping(pure_path,['packetSize','apppType']))

mixed_path = (df['source'] == 'Asia')
print(checkStopping(mixed_path,[]))

empty_path = (df['source'] == 'US') & (df['whiteListedIP'] == 'Yes') & (df['packetSize'] == 'Small')
print(checkStopping(empty_path,['appType']))

print(checkStopping(mixed_path,['packetSize','appType']))

stop_pure
no_attributes_left
stop_empty
continue


In [34]:
def main(df_input):
    target_column = 'malicious'
    feature_list = [col for col in df.columns if col != target_column]
    all_paths = []
    def growTree(current_mask, current_path, remaining_atts):
        status = checkStopping(current_mask, remaining_atts)
        if status != 'continue':
            count = divideByLabel(current_mask)
            prediction = 'Y' if count['Y'] >= count['N'] else 'N'
            leaf_summary = 'predict:' + prediction + ' ' + str(count)
            all_paths.append(current_path + [leaf_summary])
            return
        split_att = findSplitAtt(current_mask, remaining_atts)
        next_atts = [att for att in remaining_atts if att != split_att]

        for val in createBranches(split_att):
            sub_mask = current_mask & (df[split_att] == val)
            sub_path = current_path + [split_att + " == '" + str(val) + "'"]
            growTree(sub_mask, sub_path, next_atts)
    root_mask = pd.Series([True] * len(df))
    growTree(root_mask, [], feature_list)
    return all_paths

In [35]:
df_data = pd.read_csv('DT_coding_sample_training.csv')
tree_paths = main(df_data)

for path in tree_paths:
    print(path)

["source == 'Asia'", "whiteListedIP == 'Yes'", "predict:Y {'Y': 3, 'N': 0}"]
["source == 'Asia'", "whiteListedIP == 'No'", "packetSize == 'Small'", "predict:N {'Y': 0, 'N': 1}"]
["source == 'Asia'", "whiteListedIP == 'No'", "packetSize == 'Large'", "predict:Y {'Y': 0, 'N': 0}"]
["source == 'Asia'", "whiteListedIP == 'No'", "packetSize == 'Medium'", "appType == 'HTTP'", "predict:Y {'Y': 1, 'N': 1}"]
["source == 'Asia'", "whiteListedIP == 'No'", "packetSize == 'Medium'", "appType == 'FTP'", "predict:Y {'Y': 0, 'N': 0}"]
["source == 'US'", "predict:N {'Y': 0, 'N': 3}"]


In [45]:
#2.
import math
def sigmoid(z):
    return 1.0/(1.0 + math.exp(-z))
x1 = 0.2
x2 = 0.5
w13 = 0.05
w14 = -0.01
w15 = 0.02
w23 = 0.01
w24 = 0.03
w25 = -0.01
w36 = 0.01
w46 = 0.05
w56 = 0.015
theta3 = -0.3
theta4 = 0.2
theta5 = 0.05
theta6 = -0.015

net3 = (w13 * x1) + (w23 * x2) + theta3
a3 = sigmoid(net3)
net4 = (w14 * x1) + (w24 * x2) + theta4
a4 = sigmoid(net4)
net5 = (w15 * x1) + (w25 * x2) + theta5
a5 = sigmoid(net5)
net6 = (w36 * a3) + (w46 * a4) + (w56 * a5) + theta6
a6 = sigmoid(net6)

print('Node 3:')
print('Net 3 =', net3)
print('Activation a3 =', a3)

print('Node 4:')
print('Net 4 =', net4)
print('Activation a4 =', a4)

print('Node 5:')
print('Net 5 =', net5)
print('Activation a5 =', a5)

print('Node 6:')
print('Net 6 =', net6)
print('Activation a6 =', a6)

Node 3:
Net 3 = -0.285
Activation a3 = 0.4292283881055083
Node 4:
Net 4 = 0.21300000000000002
Activation a4 = 0.553049584279497
Node 5:
Net 5 = 0.049
Activation a5 = 0.5122475495675138
Node 6:
Net 6 = 0.02462847633854264
Activation a6 = 0.5061568078807185
